In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock

# Chargement

In [ ]:
# Comptabilité
df = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])
# DF Loyers
loyers = df.query("code_compte == '70600000'")
# Colonne "canal"
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

# AirBNB
airbnb = pd.read_csv(r"..\data\processed\airbnb.csv", sep=';', decimal=',', parse_dates=['mois'])
airbnb['mois'] = airbnb['mois'].dt.to_period('M')

# Préparation

In [ ]:
ca_ptf = loyers.groupby([loyers['date_écriture'].dt.to_period('M'), 'canal'])[['crédit_euro', 'débit_euro']].sum()
ca_ptf = ca_ptf.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_ptf['periode'] = ca_ptf['date_écriture'].astype(str)

In [ ]:
sherlock(ca_ptf)

In [ ]:
sherlock(airbnb)

# Merge

In [ ]:
df_merge = ca_ptf.merge(airbnb, left_on='date_écriture', right_on='mois', how='left')

In [ ]:
df_merge.head()